In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

In [ ]:
class Mission6(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=40)
        self.goals = [
            GoalCircle((0.7, 1.45), 0.35, should_stop=False),
            GoalCircle((0.25, 2.23), 0.35, should_stop=False),
            GoalCircle((-0.2, 1.45), 0.35, should_stop=False),
            GoalCircle((0.7, 1.45), 0.35, should_stop=False),
        ]
        self.initial_xy = (-0.65, 1.45)
        self.random_d_xy = (0.05, 0.05)
        self.random_d_yaw_deg = 5
        self.set_signs(
            [
                Sign(x=-0.2, y=1.45, name="original"),
                Sign(x=0.7, y=1.45, name="original"),
                Sign(x=0.25, y=2.23, name="original"),
            ]
        )

    @staticmethod
    def command_func(alive, *, move, rotate, search, **kwargs):
        """回答例：標識に向かって進み、標識が見つからなくなったら左回転して次の標識を探す"""
        while alive():
            pos = search(name="original")
            if pos is None:
                # ####### ここから下に「標識が見つからなかった場合」のプログラムを書こう
                rotate(w=45)
                ######## ここより上にプログラムを書こう
            else:
                if pos.theta > 5:
                    rotate(w=45)
                elif pos.theta < -5:
                    rotate(w=-45)
                else:
                    move(v=0.2)


sim = CarSim(prop, Mission6())
sim.run()
SimDrawer(sim).show()

In [ ]:
# 複数回実行して必ず成功するかのチェック
CHECK_CNT = 10
ok = 0
for i in range(CHECK_CNT):
    mission = Mission6()
    sim = CarSim(prop, mission)
    complete = sim.run()
    if complete:
        goal_cnt = sim.history.goal_cnt[-1]
        goals = len(mission.goals)
        goaled = goal_cnt == goals
        if goaled:
            ok += 1
        else:
            SimDrawer(sim).show()
            break
        print("-------------------------------OK:", ok)
